# Install & Imports

In [ ]:
!pip install pandas numpy scikit-learn joblib python-docx --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.2 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
from datetime import datetime
from collections import Counter
import json
import joblib
from docx import Document
from sklearn.preprocessing import LabelEncoder

# Mount Drive and Set Paths

In [ ]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


In [ ]:
BASE = "/users/"

DS1 = os.path.join(BASE, "CIC_IIoT_2025")
DS2 = os.path.join(BASE, "CIC_BCCC_NRC_IoMT_2024")
DS3 = os.path.join(BASE, "CSE_CIC_IDS2018")

OUT_DIR = os.path.join(BASE, "Combined")
os.makedirs(OUT_DIR, exist_ok=True)

print("Output Folder:", OUT_DIR)

Output Folder: /users/


# Load Latent Arrays + Labels

In [ ]:
def load_latent_dataset(folder):
    X_train = np.load(os.path.join(folder, "train_latent.npy"))
    X_val   = np.load(os.path.join(folder, "val_latent.npy"))
    X_test  = np.load(os.path.join(folder, "test_latent.npy"))

    y_train = np.load(os.path.join(folder, "y_train.npy"))
    y_val   = np.load(os.path.join(folder, "y_val.npy"))
    y_test  = np.load(os.path.join(folder, "y_test.npy"))

    return X_train, X_val, X_test, y_train, y_val, y_test

print("Loading datasets...")
d1 = load_latent_dataset(DS1)
d2 = load_latent_dataset(DS2)
d3 = load_latent_dataset(DS3)

print("✔ All latent embeddings loaded.")

Loading datasets...
✔ All latent embeddings loaded.


# Load Dataset-Wise Label Encoders

In [ ]:
def load_encoder(folder):
    p = os.path.join(folder, "label_encoder.pkl")
    if os.path.exists(p):
        return joblib.load(p)
    else:
        raise FileNotFoundError(f"Missing label encoder: {p}")

enc1 = load_encoder(DS1)
enc2 = load_encoder(DS2)
enc3 = load_encoder(DS3)

print("✔ Loaded encoders for all datasets.")

✔ Loaded encoders for all datasets.


# Safe Inverse Transform Function

In [ ]:
def safe_inverse_transform(encoder, y):
    try:
        inv = encoder.inverse_transform(y)
    except:
        inv = [str(x) for x in y]

    # Force every label to be string
    return np.array([str(x) for x in inv], dtype="object")

# Convert All Labels to Attack Names

In [ ]:
# CIC_IIoT_2025
y1_train_names = safe_inverse_transform(enc1, d1[3])
y1_val_names   = safe_inverse_transform(enc1, d1[4])
y1_test_names  = safe_inverse_transform(enc1, d1[5])

# IoMT_2024
y2_train_names = safe_inverse_transform(enc2, d2[3])
y2_val_names   = safe_inverse_transform(enc2, d2[4])
y2_test_names  = safe_inverse_transform(enc2, d2[5])

# CICIDS2018
y3_train_names = safe_inverse_transform(enc3, d3[3])
y3_val_names   = safe_inverse_transform(enc3, d3[4])
y3_test_names  = safe_inverse_transform(enc3, d3[5])

print("✔ Converted all dataset labels to attack names (strings).")

✔ Converted all dataset labels to attack names (strings).


# Build Global Class Space

In [ ]:
all_names = np.concatenate([
    y1_train_names, y1_val_names, y1_test_names,
    y2_train_names, y2_val_names, y2_test_names,
    y3_train_names, y3_val_names, y3_test_names
])

all_names = all_names.astype("object")

# Remove invalid entries
all_names = all_names[all_names != "nan"]
all_names = all_names[all_names != "None"]

global_classes = np.unique(all_names)
print("Global classes:", len(global_classes))

Global classes: 37


# Fit Global Encoder

In [ ]:
global_encoder = LabelEncoder()
global_encoder.fit(global_classes)

joblib.dump(global_encoder, os.path.join(OUT_DIR, "combined_label_encoder.pkl"))

print("✔ Saved global label encoder with", len(global_classes), "classes.")

✔ Saved global label encoder with 37 classes.


# Convert All Labels to Global IDs

In [ ]:
y_train_global = np.concatenate([
    global_encoder.transform(y1_train_names),
    global_encoder.transform(y2_train_names),
    global_encoder.transform(y3_train_names)
])

y_val_global = np.concatenate([
    global_encoder.transform(y1_val_names),
    global_encoder.transform(y2_val_names),
    global_encoder.transform(y3_val_names)
])

y_test_global = np.concatenate([
    global_encoder.transform(y1_test_names),
    global_encoder.transform(y2_test_names),
    global_encoder.transform(y3_test_names)
])

print("✔ Converted all labels into global unified label space.")

✔ Converted all labels into global unified label space.


# Combine Latent Embeddings

In [ ]:
X_train = np.vstack([d1[0], d2[0], d3[0]])
X_val   = np.vstack([d1[1], d2[1], d3[1]])
X_test  = np.vstack([d1[2], d2[2], d3[2]])

print("Combined X_train:", X_train.shape)
print("Combined X_val:", X_val.shape)
print("Combined X_test:", X_test.shape)

Combined X_train: (9136746, 64)
Combined X_val: (1957874, 64)
Combined X_test: (1957876, 64)


# Save Combined Dataset

In [ ]:
np.save(os.path.join(OUT_DIR, "combined_train_latent.npy"), X_train)
np.save(os.path.join(OUT_DIR, "combined_val_latent.npy"),   X_val)
np.save(os.path.join(OUT_DIR, "combined_test_latent.npy"),  X_test)

np.save(os.path.join(OUT_DIR, "combined_y_train.npy"), y_train_global)
np.save(os.path.join(OUT_DIR, "combined_y_val.npy"),   y_val_global)
np.save(os.path.join(OUT_DIR, "combined_y_test.npy"),  y_test_global)

print("✔ Saved combined dataset successfully.")

✔ Saved combined dataset successfully.


# Save Summary JSON + DOCX

In [ ]:
def convert_keys_to_int(d):
    return {int(k): int(v) for k, v in d.items()}

summary = {
    "generated_at": datetime.now().isoformat(),
    "latent_dim": int(X_train.shape[1]),
    "num_classes": int(len(global_classes)),
    "sizes": {
        "train": int(len(X_train)),
        "val":   int(len(X_val)),
        "test":  int(len(X_test)),
    },
    "class_distribution": {
        "train": convert_keys_to_int(Counter(y_train_global)),
        "val":   convert_keys_to_int(Counter(y_val_global)),
        "test":  convert_keys_to_int(Counter(y_test_global)),
    }
}

with open(os.path.join(OUT_DIR, "combined_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("✔ JSON summary saved successfully!")

✔ JSON summary saved successfully!


In [ ]:
doc = Document()
doc.add_heading("FALCON-ID Combined Dataset Summary", level=1)
doc.add_paragraph(f"Generated at: {datetime.now()}")

doc.add_heading("Dataset Sizes", level=2)
for k,v in summary["sizes"].items():
    doc.add_paragraph(f"{k}: {v}")

doc.add_heading("Class Distribution", level=2)
for split, dist in summary["class_distribution"].items():
    doc.add_heading(split, level=3)
    for cls, cnt in dist.items():
        doc.add_paragraph(f"{cls}: {cnt}", style="List Bullet")

doc.save(os.path.join(OUT_DIR, "combined_summary.docx"))

print("✔ DOCX summary saved.")

✔ DOCX summary saved.
